# RoBERTa Sentiment Analysis - Amazon Reviews

Finalny model transformatorowy do porownania z poprzednimi eksperymentami:
- **SVM + TF-IDF**
- **BiLSTM + GloVe**
- **RoBERTa / DistilRoBERTa**

Notebook jest ustawiony tak, zeby byl mozliwy do uruchomienia na komputerze z ok. 16 GB RAM:
- domyslnie uzywa `distilroberta-base`, czyli lzejszego wariantu RoBERTy,
- tokenizuje teksty w batchach w `collate_fn`, a nie zapisuje calego zakodowanego zbioru w RAM,
- uzywa malego batch size i gradient accumulation,
- ogranicza dlugosc sekwencji do 128 tokenow,
- ma early stopping na zbiorze walidacyjnym.

Jesli komputer poradzi sobie bez problemu, mozna zmienic `MODEL_NAME` na `roberta-base`, ale pierwszy przebieg bezpieczniej zaczac od `distilroberta-base`.

## 1. Importy i ustawienia

In [1]:
import os
import re
import time
import copy
import random
import gc
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import seaborn as sns
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

# Zostawiamy troche oddechu systemowi przy treningu na CPU.
if DEVICE.type == 'cpu':
    torch.set_num_threads(max(1, (os.cpu_count() or 2) - 2))

print(f'Urzadzenie: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Urzadzenie: mps
PyTorch: 2.11.0


## 2. Konfiguracja eksperymentu

Ustawienia ponizej sa dobrane konserwatywnie pod 16 GB RAM. Najwazniejsze pokretla:
- `MODEL_NAME`: `distilroberta-base` jest najbezpieczniejsze; `roberta-base` da mocniejszy, ale ciezszy eksperyment.
- `MAX_LENGTH`: 128 trzyma pamiec w ryzach. 256 moze poprawic wynik przy dlugich recenzjach, ale mocno zwieksza koszt.
- `LIMIT_TRAIN_PER_CLASS`: ustaw np. `3000`, jesli chcesz szybki test techniczny. Dla finalnego wyniku zostaw `None`.

In [2]:
MODEL_NAME = 'distilroberta-base'
# MODEL_NAME = 'roberta-base'  # mocniejszy, ale znacznie ciezszy wariant

MAX_LENGTH = 128
NUM_CLASSES = 3
TARGET_NAMES = ['Negative (0)', 'Neutral (1)', 'Positive (2)']

# None = pelny zbior treningowy. Liczba = szybki test, np. 3000 probek na klase tylko z train.
LIMIT_TRAIN_PER_CLASS = None

if DEVICE.type == 'cuda':
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 4
    USE_AMP = True
elif DEVICE.type == 'mps':
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 8
    USE_AMP = False
else:
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 8
    USE_AMP = False

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 4
PATIENCE = 1
MAX_GRAD_NORM = 1.0

print('Konfiguracja:')
print(f'  MODEL_NAME: {MODEL_NAME}')
print(f'  MAX_LENGTH: {MAX_LENGTH}')
print(f'  TRAIN_BATCH_SIZE: {TRAIN_BATCH_SIZE}')
print(f'  EVAL_BATCH_SIZE: {EVAL_BATCH_SIZE}')
print(f'  GRADIENT_ACCUMULATION_STEPS: {GRADIENT_ACCUMULATION_STEPS}')
print(f'  Efektywny batch size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}')
print(f'  LIMIT_TRAIN_PER_CLASS: {LIMIT_TRAIN_PER_CLASS}')

Konfiguracja:
  MODEL_NAME: distilroberta-base
  MAX_LENGTH: 128
  TRAIN_BATCH_SIZE: 4
  EVAL_BATCH_SIZE: 8
  GRADIENT_ACCUMULATION_STEPS: 8
  Efektywny batch size: 32
  LIMIT_TRAIN_PER_CLASS: None


## 3. Wczytanie i lekkie czyszczenie danych

In [3]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

DATA_PATH = 'data/processed_reviews.csv'
df = pd.read_csv(DATA_PATH)
df['clean_text'] = df['full_text'].map(clean_text)
df['label'] = df['label'].astype(int)

print(f'Liczba probek: {len(df)}')
print(f'Kolumny: {df.columns.tolist()}')
print('\nRozklad klas:')
print(df['label'].value_counts().sort_index())

df[['clean_text', 'score', 'sentiment', 'label']].head(3)

Liczba probek: 89511
Kolumny: ['id', 'full_text', 'score', 'sentiment', 'label', 'clean_text']

Rozklad klas:
label
0    29837
1    29837
2    29837
Name: count, dtype: int64


,clean_text,score,sentiment,label
0,It's not equal amounts of each flavor... Just ...,3,0,1
1,Not A Gingerbread Man These do taste very good...,4,1,2
2,"great taste great taste, good price... Always ...",5,1,2


## 4. Podzial train / validation / test

Uzywamy takiego samego schematu jak w notebooku BiLSTM:
- 70% train,
- 10% validation,
- 20% test.

Test zostaje odlozony do finalnej ewaluacji.

In [4]:
X = df['clean_text']
y = df['label']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=SEED, stratify=y_temp
)

if LIMIT_TRAIN_PER_CLASS is not None:
    train_df = pd.DataFrame({'text': X_train, 'label': y_train})
    train_df = (
        train_df.groupby('label', group_keys=False)
        .apply(lambda g: g.sample(n=min(LIMIT_TRAIN_PER_CLASS, len(g)), random_state=SEED))
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    X_train = train_df['text']
    y_train = train_df['label']

print(f'Train:      {len(X_train):>6} probek ({len(X_train)/len(X)*100:.1f}%)')
print(f'Validation: {len(X_val):>6} probek ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test:       {len(X_test):>6} probek ({len(X_test)/len(X)*100:.1f}%)')

print('\nRozklad klas w Train:')
print(y_train.value_counts().sort_index())
print('\nRozklad klas w Validation:')
print(y_val.value_counts().sort_index())
print('\nRozklad klas w Test:')
print(y_test.value_counts().sort_index())

Train:       62657 probek (70.0%)
Validation:   8951 probek (10.0%)
Test:        17903 probek (20.0%)

Rozklad klas w Train:
label
0    20886
1    20885
2    20886
Name: count, dtype: int64

Rozklad klas w Validation:
label
0    2984
1    2984
2    2983
Name: count, dtype: int64

Rozklad klas w Test:
label
0    5967
1    5968
2    5968
Name: count, dtype: int64


## 5. Dataset i DataLoader

Tokenizacja dzieje sie dynamicznie dla kazdego batcha. To jest wolniejsze niz zapisanie tokenow raz na zawsze, ale duzo lagodniejsze dla RAM.

In [5]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels.astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

@dataclass
class RobertaCollator:
    tokenizer: object
    max_length: int

    def __call__(self, batch):
        texts, labels = zip(*batch)
        encoded = self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        encoded['labels'] = torch.tensor(labels, dtype=torch.long)
        return encoded

collator = RobertaCollator(tokenizer=tokenizer, max_length=MAX_LENGTH)

train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)
test_dataset = TextDataset(X_test, y_test)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    collate_fn=collator,
    num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)

print(f'Train batche:      {len(train_loader)}')
print(f'Validation batche: {len(val_loader)}')
print(f'Test batche:       {len(test_loader)}')

Train batche:      15665
Validation batche: 1119
Test batche:       2238


## 6. Model RoBERTa

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
    label2id={'negative': 0, 'neutral': 1, 'positive': 2}
)
model.to(DEVICE)

# Dla klasyfikacji nie potrzebujemy cache dekodera; wylaczenie bywa pomocne pamieciowo.
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.__class__.__name__)
print(f'Laczna liczba parametrow: {total_params:,}')
print(f'Trenowalne parametry:     {trainable_params:,}')

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification
Laczna liczba parametrow: 82,120,707
Trenowalne parametry:     82,120,707


/var/folders/4p/1fp7f9tn50qd5dg43z45lvz00000gn/T/ipykernel_97792/523077928.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 7. Funkcje treningu i ewaluacji

In [7]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch, DEVICE)
        labels = batch['labels']

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(**batch)
            loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        preds = outputs.logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        del batch, outputs, loss

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)
        labels = batch['labels']
        outputs = model(**batch)
        loss = outputs.loss

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        del batch, outputs, loss

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1, all_preds, all_labels

print('Funkcje treningowe zdefiniowane.')

Funkcje treningowe zdefiniowane.


## 8. Trening z early stopping

Najlepszy model jest wybierany po F1-macro na walidacji. Przy 16 GB RAM celowo trzymamy malo epok - transformery zwykle ucza sie szybko na takim zadaniu.

In [8]:
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

best_val_f1 = -1.0
best_model_wts = None
best_epoch = 0
epochs_no_improve = 0

print(f'Trening {MODEL_NAME} - max {MAX_EPOCHS} epoki, patience={PATIENCE}')
print('=' * 75)

start = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss, train_f1 = train_epoch(model, train_loader, optimizer)
    val_loss, val_f1, _, _ = evaluate(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)

    epoch_time = time.time() - t0
    print(f'Epoka {epoch:>2}/{MAX_EPOCHS} | '
          f'Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | '
          f'Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | '
          f'{epoch_time:.0f}s')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_model_wts = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping po epoce {epoch} (brak poprawy F1 przez {PATIENCE} epoke/epoki)')
            break

    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

if best_model_wts is not None:
    model.load_state_dict(best_model_wts)
    model.to(DEVICE)

print(f'\nCalkowity czas treningu: {time.time() - start:.0f}s')
print(f'Najlepsza epoka: {best_epoch} (val F1: {best_val_f1:.4f})')

/var/folders/4p/1fp7f9tn50qd5dg43z45lvz00000gn/T/ipykernel_97792/4014740240.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Trening distilroberta-base - max 4 epoki, patience=1
Epoka  1/4 | Train loss: 0.5350 | Val loss: 0.4674 | Train F1: 0.7728 | Val F1: 0.8062 | 2879s


/var/folders/4p/1fp7f9tn50qd5dg43z45lvz00000gn/T/ipykernel_97792/4014740240.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoka  2/4 | Train loss: 0.4317 | Val loss: 0.4543 | Train F1: 0.8216 | Val F1: 0.8087 | 2814s


/var/folders/4p/1fp7f9tn50qd5dg43z45lvz00000gn/T/ipykernel_97792/4014740240.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


KeyboardInterrupt: 

## 9. Krzywe uczenia

In [ ]:
epochs_ran = len(history['train_loss'])
ep = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(ep, history['train_loss'], label='Train loss', marker='o', markersize=4)
ax1.plot(ep, history['val_loss'], label='Val loss', marker='o', markersize=4)
ax1.axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Najlepsza epoka ({best_epoch})')
ax1.set_xlabel('Epoka')
ax1.set_ylabel('Loss')
ax1.set_title('Krzywe strat')
ax1.legend()

ax2.plot(ep, history['train_f1'], label='Train F1', marker='o', markersize=4)
ax2.plot(ep, history['val_f1'], label='Val F1', marker='o', markersize=4)
ax2.axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Najlepsza epoka ({best_epoch})')
ax2.set_xlabel('Epoka')
ax2.set_ylabel('F1-macro')
ax2.set_title('Krzywe F1-macro')
ax2.legend()

plt.tight_layout()
plt.show()

if best_epoch > 0:
    gap = history['train_f1'][best_epoch - 1] - history['val_f1'][best_epoch - 1]
    print(f'Gap train-val F1 w najlepszej epoce: {gap:+.4f}', '<-- ok' if gap < 0.05 else '<-- uwaga, mozliwe przeuczenie')

## 10. Wyniki walidacyjne

In [ ]:
_, val_f1_final, val_preds, val_labels = evaluate(model, val_loader)

print('=' * 55)
print('WYNIKI NA ZBIORZE WALIDACYJNYM')
print('=' * 55)
print(f'F1-macro (validation): {val_f1_final:.4f}')
print()
print(classification_report(val_labels, val_preds, target_names=TARGET_NAMES))

cm_val = confusion_matrix(val_labels, val_preds)
cm_val_pct = cm_val.astype(float) / cm_val.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_val_pct, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Neg (0)', 'Neu (1)', 'Pos (2)'],
            yticklabels=['Neg (0)', 'Neu (1)', 'Pos (2)'])
plt.ylabel('Etykieta rzeczywista')
plt.xlabel('Predykcja modelu')
plt.title(f'Confusion Matrix - Zbior Walidacyjny (F1={val_f1_final:.4f})')
plt.tight_layout()
plt.show()

## 11. Finalna ewaluacja na zbiorze testowym

In [ ]:
_, test_f1, test_preds, test_labels = evaluate(model, test_loader)

print('=' * 55)
print('FINALNE WYNIKI NA ZBIORZE TESTOWYM')
print('=' * 55)
print(f'Val F1-macro  (najlepsza epoka): {val_f1_final:.4f}')
print(f'Test F1-macro (FINALNA):         {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=TARGET_NAMES))

cm_test = confusion_matrix(test_labels, test_preds)
cm_test_pct = cm_test.astype(float) / cm_test.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_test_pct, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=['Neg (0)', 'Neu (1)', 'Pos (2)'],
            yticklabels=['Neg (0)', 'Neu (1)', 'Pos (2)'])
plt.ylabel('Etykieta rzeczywista')
plt.xlabel('Predykcja modelu')
plt.title(f'Confusion Matrix - Zbior Testowy (F1={test_f1:.4f})')
plt.tight_layout()
plt.show()

## 12. Porownanie z poprzednimi modelami

Wartosc BiLSTM uzupelnij, jesli po ponownym uruchomieniu notebooka `bilstm_glove_sentiment_final.ipynb` wyszla inna liczba. Aktualnie w projekcie SVM ma `0.7761`, a BiLSTM bylo ok. `0.7841` na podstawie zapisanego porownania.

In [ ]:
svm_test_f1 = 0.7761
bilstm_test_f1 = 0.7841

comparison_df = pd.DataFrame({
    'model': ['SVM + TF-IDF', 'BiLSTM + GloVe', MODEL_NAME],
    'test_f1_macro': [svm_test_f1, bilstm_test_f1, test_f1]
})
comparison_df['delta_vs_svm'] = comparison_df['test_f1_macro'] - svm_test_f1
comparison_df['delta_vs_bilstm'] = comparison_df['test_f1_macro'] - bilstm_test_f1

display(comparison_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_df, x='model', y='test_f1_macro', palette=['steelblue', 'seagreen', 'indianred'])
plt.ylim(0, 1)
plt.ylabel('F1-macro na zbiorze testowym')
plt.xlabel('Model')
plt.title('Porownanie finalnych wynikow testowych')
for i, score in enumerate(comparison_df['test_f1_macro']):
    plt.text(i, score + 0.01, f'{score:.4f}', ha='center')
plt.tight_layout()
plt.show()

print(f'Roznica {MODEL_NAME} - SVM:    {test_f1 - svm_test_f1:+.4f} F1-macro')
print(f'Roznica {MODEL_NAME} - BiLSTM: {test_f1 - bilstm_test_f1:+.4f} F1-macro')

## 13. Podsumowanie eksperymentu

In [ ]:
print('=' * 55)
print('PODSUMOWANIE EKSPERYMENTU')
print('=' * 55)
print(f'\nModel:')
print(f'  Transformer:        {MODEL_NAME}')
print(f'  Max dlugosc seq.:   {MAX_LENGTH} tokenow')
print(f'  Liczba klas:        {NUM_CLASSES}')
print(f'\nPodzial danych:')
print(f'  Train:              {len(X_train):>6} probek')
print(f'  Validation:         {len(X_val):>6} probek')
print(f'  Test:               {len(X_test):>6} probek')
print(f'\nTrening:')
print(f'  Batch size:         {TRAIN_BATCH_SIZE}')
print(f'  Gradient accum.:    {GRADIENT_ACCUMULATION_STEPS}')
print(f'  Efektywny batch:    {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}')
print(f'  Learning rate:      {LEARNING_RATE}')
print(f'  Najlepsza epoka:    {best_epoch}/{epochs_ran}')
print(f'\nWyniki:')
print(f'  Val F1-macro:       {val_f1_final:.4f}')
print(f'  Test F1-macro:      {test_f1:.4f}')